# Project 3 - NCSU ST 554
#### Author: Trevor Lillywhite
#### Due Date: April 30, 2026

## Introduction

**Purpose:** This project will focus on using `spark` to handle streaming data, fitting a machine learning model, and performing other associated tasks. 

It will involve both this notebook (using a pySpark3 kernel) and producing a `.py` file to read in a stream of data from a static source. After a portion of the dataset is used for model training, the remaining portions will be incrementally streamed in the form of csv files to a folder being monitored by the code. The machine learning model will do prediction on this stream and write the results out to the console.

**Data Source:** The data is adapted from a UCI machine learning dataset on a study predicting power consumption to factors like time of day, temperature, and humidity. More information on the original dataset is available here: 
+ UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city
+ Original Case Study: https://www.semanticscholar.org/paper/Comparison-of-Machine-Learning-Algorithms-for-the-%3A-Salam-Hibaoui/177512e766fe5d8624a1b3e93abd11082ac37b3f


## Fitting the Model

### Reading in Data

The data is saved as a csv in the same directory level as this notebook. However, it can also be accessed at the following URL: 
+ https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv

We will read in the data as a standard `pandas` DataFrame using the `pd.read_csv()` function.

In [2]:
# Import relevant modules
import pandas as pd
import numpy as np
import time
from pyspark.sql.types import StructType
from pyspark.sql import SparkSession

# Create spark session
spark = SparkSession.builder.getOrCreate()

In [3]:
# Read in data to pandas DataFrame
df_pandas = pd.read_csv('power_ml_data.csv', sep=',', header=0)

# Verify data
df_pandas.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [4]:
# Check dataframe info
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47174 entries, 0 to 47173
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Temperature            47174 non-null  float64
 1   Humidity               47174 non-null  float64
 2   Wind_Speed             47174 non-null  float64
 3   General_Diffuse_Flows  47174 non-null  float64
 4   Diffuse_Flows          47174 non-null  float64
 5   Power_Zone_1           47174 non-null  float64
 6   Power_Zone_2           47174 non-null  float64
 7   Power_Zone_3           47174 non-null  float64
 8   Month                  47174 non-null  int64  
 9   Hour                   47174 non-null  int64  
dtypes: float64(8), int64(2)
memory usage: 3.6 MB


It appears that the data was read in properly. All columns are numerical (`float64` or `int64`) with no missing values (47174 non-null values for each column). 

Now we will convert this to a `spark` DataFrame.

In [6]:
# Convert to spark DataFrame
df = spark.createDataFrame(df_pandas)
df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

The data appears consistent after converting to `spark`. 

Let's check the data types of each column. 

In [7]:
# Check spark DataFrame data types (displayed cleanly in Pandas)
pd.DataFrame(df.dtypes)

,0,1
0,Temperature,double
1,Humidity,double
2,Wind_Speed,double
3,General_Diffuse_Flows,double
4,Diffuse_Flows,double
5,Power_Zone_1,double
6,Power_Zone_2,double
7,Power_Zone_3,double
8,Month,bigint
9,Hour,bigint


Each `float64` value converted to `double` and each `int64` value converted to `bigint`.

### Data Preparation

Our analysis will treat `Power_Zone_3` as the response variable and all other variables as predictors. 

We want to fit an Elastic Net model using cross validation. Because the actual predictions will be performed on a separate streaming dataset, we don't need to do a test/train split on the data we just read in. Data preparation steps will be executed using `MLlib` transformations assembled into a pipeline.

The pipeline will contain following transformations: 
+ Cast `Hour` to `double` type, if not already done. 
+ Binarize the `Hour` column based on the column being less than 6.5 or not (to discriminate nighttime electrical loads from daytime). 
+ One-hot encode the `Month` column.
+ Run a PCA fit on the `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` columns. 
    - This will find a number of aggregated features (using linear combinations of those listed above) that are orthogonal (uncorrelated) and able to be sorted by importance by eigenvalues. 
    - The features will be bundled together using `VectorAssembler()`
    - We will fit the `PCA()` estimator using the feature vector. 
    - Only the two most important Principal Components (PCs) will be used in the pipeline. This will reduce the dimensionality of the data while accounting for most of the variance.
+ The response variable (`Power_Zone_3`) will be renamed to `label`.
+ The two PCs and remaining features will be bundled together using `VectorAssembler()`

In [8]:
# Import relevant modules
from pyspark.sql.functions import col
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, StandardScaler, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

In [9]:
# Create SQL Transformer to cast Hour to double type
sqlCast = SQLTransformer(
    statement = """
                SELECT * EXCEPT (Hour), CAST(Hour AS DOUBLE) 
                FROM __THIS__
                """
)

In [10]:
# Binarize Hour based on threshold of 6.5
binarizer = Binarizer(threshold = 6.5, inputCol = "Hour", outputCol = "Hour_binarized")

In [11]:
# One-hot encode Month column
encoder = OneHotEncoder(inputCols=["Month"], 
                        outputCols=["Month_encoded"])

In [12]:
# Run PCA fit select PCs

# Define features of interest
colsPCA = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"]

# Assemble features into vector
assemblerPCA = VectorAssembler(inputCols = colsPCA, outputCol = "featuresPCA")

# Standardize features: mean=0, std=1. 
scalerPCA = StandardScaler(inputCol = "featuresPCA", 
                           outputCol = "scaledFeaturesPCA", 
                           withStd = True, withMean = True)

# Create PCA transformer
pca = PCA(k=2, inputCol = "scaledFeaturesPCA", outputCol = "pcaFeatures")

Note that after fitting PCA, we can check model.stages[pcaIndex].explainedVariance to see how much information each PC retained

Let's test this and the general PCA fitting/transformation process

In [14]:
# Test PCA fitting and PC importance
pipeTest = Pipeline(stages=[assemblerPCA, scalerPCA, pca])
modelPCA = pipeTest.fit(df)
modelPCA.stages[2].explainedVariance

DenseVector([0.4658, 0.2387])

The PC1 explains nearly half of the variance, and PC2 explains nearly a quarter. 

In the `pca` transformer definition, I also experimented by changing k to equal 5. The remaining components had importances of 0.1438, 0.0837, and 0.0681. It makes sense to only use the first two PCs since they capture about 70% of the total variance and there are diminishing returns after that.

In [16]:
# Complete the PCA fitting test
resultsTest = modelPCA.transform(df)
resultsTestPandas = resultsTest.toPandas()
resultsTestPandas.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,featuresPCA,scaledFeaturesPCA,pcaFeatures
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,"[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.069074321346372, 0.8078678829472271]"
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,"[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.102921063880654, 0.8152476222806402]"
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,"[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.1120662633791016, 0.8227151924829966]"
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,"[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.1435031847422197, 0.8329135817940974]"
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,"[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.1824440036627006, 0.8444681624812338]"


The new columns were produced as directed. The values look credible. 

Let's get back on track with the transformation definitions. 

In [17]:
# Rename response variable as 'label'
sqlRename = SQLTransformer(
    statement = """ 
                SELECT * EXCEPT (Power_Zone_3), (Power_Zone_3) AS label
                FROM __THIS__
                """
)

In [18]:
# Assemble all predictors into vector
assembler = VectorAssembler(inputCols = ["pcaFeatures", "Hour_binarized", "Power_Zone_1", "Power_Zone_2", "Month_encoded"], 
                            outputCol = "features")

In [21]:
# Now assemble into pipeline and check the outputs after fitting
pipelineTransformers = Pipeline(stages = [sqlCast, binarizer, encoder, assemblerPCA, scalerPCA, pca, sqlRename, assembler])

test = pipeline.fit(df)
test.transform(df).show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+--------------+--------------+--------------------+--------------------+--------------------+-----------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|Hour_binarized| Month_encoded|         featuresPCA|   scaledFeaturesPCA|         pcaFeatures|      label|            features|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+--------------+--------------+--------------------+--------------------+--------------------+-----------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|    1| 0.0|           0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[-2.1079477438809...|[2.06907432134637...|20240.96386|(17,[0,1,3,4,6],[...|
|      6.414|    74.5|     0.083|                 0.07|        0.085

The pipelines appears to have executed correctly. 

### Model Fitting

The pipeline defined above will be used in a `CrossValidator()` function to fit an elastic net estimator using `LinearRegression()` function. We will explore a grid of the following parameters (with all combinations): 
+ `regParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
+ `elasticNetParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

The `CrossValidator()` will use `rmse` as the model selection metric. 

In [30]:
# Instantiate model
en = LinearRegression()

# Add estimator to pipeline
pipeline = Pipeline(stages = [sqlCast, binarizer, encoder, assemblerPCA, scalerPCA, pca, sqlRename, assembler, en])

# Set up parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(en.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(en.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Set up Cross Validator
cv = CrossValidator(estimator = pipeline,
                    estimatorParamMaps = paramGrid,
                    evaluator = RegressionEvaluator(metricName = "rmse"),
                    numFolds = 5,
                    seed = 42,
                    parallelism = 128)

In [31]:
# Fit Cross Validator
enModel = cv.fit(df)

26/04/24 21:12:56 WARN Instrumentation: [35575ae9] regParam is zero, which might cause numerical instability and overfitting.
26/04/24 21:13:01 WARN Instrumentation: [35575ae9] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/24 21:13:01 WARN Instrumentation: [2c338631] regParam is zero, which might cause numerical instability and overfitting.
26/04/24 21:13:02 WARN Instrumentation: [761629ea] regParam is zero, which might cause numerical instability and overfitting.
26/04/24 21:13:02 WARN Instrumentation: [abec3085] regParam is zero, which might cause numerical instability and overfitting.
26/04/24 21:13:02 WARN Instrumentation: [4bccfecf] regParam is zero, which might cause numerical instability and overfitting.
26/04/24 21:13:03 WARN Instrumentation: [5f91c630] regParam is zero, which might cause numerical instability and overfitting.
26/04/24 21:13:03 WARN Instrumentation: [592e232f] regParam is zero, which might cause numerical ins

Note that, without setting the `parallelism` argument, cross-validation is performed fully in series. Here, it took about 12 minutes to explore the hyperparameter space using a single logical processor. After setting `parallelism` to 128, fitting took only 5 minutes. 

Let's check out the optimal values chosen for the tuning parameters and CV error.

In [34]:
# Isolate best estimator and its parameters
bestPipelineModel = enModel.bestModel
bestENModel = bestPipelineModel.stages[-1]
print("Best regParam:", bestENModel.getRegParam())
print("Best elasticNetParam:", bestENModel.getElasticNetParam())

Best regParam: 0.05
Best elasticNetParam: 0.05


The best model had a relatively small amount of regularization overall and also a small amount of L1 regularization (mostly L2 regularization).

Next, we will report the CV error to see how well the fitting process went. These will be averaged across folds.

In [41]:
# Report CV error
print(f"Best CV RMSE: {min(enModel.avgMetrics):.4f} \n")
print(f"RMSE statistics for all parameter combinations (for comparison):")
pd.Series(enModel.avgMetrics).describe()

Best CV RMSE: 2174.9962 

RMSE statistics for all parameter combinations (for comparison):


count     121.000000
mean     2175.008106
std         0.012939
min      2174.996249
25%      2174.997948
50%      2175.000512
75%      2175.016981
max      2175.034889
dtype: float64

There wasn't much variation at all! Based on the min and max, all of the models performed almost identically.

The next step is to report the training set RMSE using the fitted model as a transformer and evaluating on the entire training set. 

In [49]:
# Determine RMSE using full training set
predictions = bestPipelineModel.transform(df)
trainRMSE = RegressionEvaluator().evaluate(predictions)
print(f"Model RMSE on Training Set: {trainRMSE:.4f}")

Model RMSE on Training Set: 2174.4490


This is slightly better than the previous value because the best model was retrained on the entire training set without a hold-out fold.

Lastly, before handling streaming data, we will create a `residual` column (`label` - `prediction`) using the `.withColumn()` method. 

In [56]:
# Create residual column
results = predictions.withColumn("residual", (col("label") - col("prediction")))\
                     .select("residual", "label", "prediction")
results.show()

+------------------+-----------+------------------+
|          residual|      label|        prediction|
+------------------+-----------+------------------+
| -695.262981006279|20240.96386| 20936.22684100628|
|1429.4978719047103|20131.08434| 18701.58646809529|
|1431.2442602264455|19668.43373|18237.189469773555|
|1283.3848937492621|18899.27711|17615.892216250737|
|1425.0277973159573|18442.40964|17017.381842684044|
|1589.5061777942356|18130.12048|16540.614302205766|
|1830.3760061054381|17945.06024| 16114.68423389456|
|1717.2387757976976|17459.27711|15742.038334202301|
|1742.9841969907102|17025.54217| 15282.55797300929|
|1859.3175485772372|16794.21687|14934.899321422763|
|1992.8155358609765|16638.07229|14645.256754139024|
|1999.5019080591956|16395.18072|14395.678811940805|
| 2043.839101080781|16117.59036| 14073.75125891922|
|2213.4085001061976| 15822.6506|13609.242099893803|
| 2240.114575045878|15672.28916|13432.174584954122|
| 2314.823027723265|15597.10843|13282.285402276735|
| 2371.43177